In [1]:
import  random

import torch
from torch.utils import data
from torchvision import transforms as T
from torch.utils.data import Subset

from src.dataloaders.dataloader_for_CNN import mavDataLoader, SequenceBatchSampler
from src.models.CNN_ResNet50_VO import CNN_ResNet50_VO
from src.models.DeepVO import DeepVO
from src.function_of_loss.mse_pose import PoseLoss
from src.piplines.pipline_for_CNN import training

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

# Нейронки

## CNN_ResNet50_VO

In [ ]:

transform = T.Compose([
    T.ToTensor()
])
dataset = mavDataLoader('datasets/euroc_mav', transform, device=device, batchsize=10) # Сразу формируем все массивы на GPU

train_size = int(0.8 * len(dataset))

dtrain = Subset(dataset, range(0, train_size))
dtest  = Subset(dataset, range(train_size, len(dataset)))

train_data = data.DataLoader(dtrain, batch_size=1, shuffle=True)
test_data = data.DataLoader(dtest, batch_size=1)
len(dataset)

In [13]:
model = CNN_ResNet50_VO()
model = model.to(device)

# всё заморозили
for p in model.parameters():
    p.requires_grad = False

# обучаем новый первый слой
for p in model.encoder.conv1.parameters():
    p.requires_grad = True

# обучаем самый верхний блок resNet
for p in model.encoder.layer4.parameters():
    p.requires_grad = True

# обучаем полносвязки
for p in model.fc1.parameters():
    p.requires_grad = True

for p in model.fc2.parameters():
    p.requires_grad = True

epochs = 10
loss_func = PoseLoss()
optimizer = torch.optim.Adam(params=filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

In [14]:
sum(p.numel() for p in model.parameters() if p.requires_grad)

17103622

In [15]:
dct_of_results = training(train_data, test_data, model, loss_func, optimizer, epochs, device=device)

Эпоха валидационная 1/10: 100%|██████████| 728/728 [03:01<00:00,  4.01it/s, loss=0.00365, Average Translational RMSE drift=0.658, Average Rotational RMSE drift=2.92, path_lenght=tensor(14.6558, device='cuda:0')]


Модель сохранена


Эпоха валидационная 2/10: 100%|██████████| 728/728 [03:04<00:00,  3.95it/s, loss=0.00217, Average Translational RMSE drift=0.364, Average Rotational RMSE drift=2.37, path_lenght=tensor(14.6558, device='cuda:0')]


Модель сохранена


Эпоха валидационная 3/10: 100%|██████████| 728/728 [02:59<00:00,  4.06it/s, loss=0.000549, Average Translational RMSE drift=0.35, Average Rotational RMSE drift=1.11, path_lenght=tensor(14.6558, device='cuda:0')]


...

Эпоха валидационная 7/10: 100%|██████████| 728/728 [02:58<00:00,  4.07it/s, loss=0.000138, Average Translational RMSE drift=0.145, Average Rotational RMSE drift=0.492, path_lenght=tensor(14.6558, device='cuda:0')]


Модель сохранена


Эпоха валидационная 8/10: 100%|██████████| 728/728 [02:59<00:00,  4.06it/s, loss=7.36e-5, Average Translational RMSE drift=0.168, Average Rotational RMSE drift=0.258, path_lenght=tensor(14.6558, device='cuda:0')]


Модель сохранена


Эпоха валидационная 10/10: 100%|██████████| 728/728 [02:57<00:00,  4.09it/s, loss=0.000114, Average Translational RMSE drift=0.143, Average Rotational RMSE drift=0.449, path_lenght=tensor(14.6558, device='cuda:0')]


In [16]:
dct_of_results

defaultdict(list,
            {'loss_train': [5.0463067141068124e-05],
             'loss_test': [0.00011364731271159671],
             'Average Translational RMSE drift': 0.1427155902959755,
             'Average Rotational RMSE drift': 0.44902428294721236})

## DeepVO

In [ ]:

transform = T.Compose([
    T.Resize((320, 192))
])
dataset = mavDataLoader('datasets/euroc_mav', transform, device='cpu', batchsize=8, hidden_size=8) # Сразу формируем все массивы на GPU

train_size = int(0.005 * len(dataset))

groups = dataset.batch_groups.copy()

train_size = int(0.8 * len(groups))
train_groups = groups[:train_size]
test_groups = groups[train_size:]


train_sampler = SequenceBatchSampler(train_groups)
test_sampler = SequenceBatchSampler(test_groups)

train_data = data.DataLoader(dataset, batch_sampler=train_sampler, num_workers=6, pin_memory=True)
test_data = data.DataLoader(dataset, batch_sampler=test_sampler, num_workers=6, pin_memory=True)
len(dataset)

36373

In [4]:
next(iter(train_data))[1].shape

torch.Size([2, 6])

In [7]:
model = DeepVO()
model = model.to(device)

epochs = 10
loss_func = PoseLoss()
optimizer = torch.optim.Adam(params=filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
sum(p.numel() for p in model.parameters() if p.requires_grad)

27958534

In [8]:
dct_of_results = training(train_data, test_data, model, loss_func, optimizer, epochs, device=device, name_of_model='DeepVO.tar', squueze=False)

Эпоха валидационная 1/10: 100%|██████████| 910/910 [04:25<00:00,  3.42it/s, loss=4.53e-5, Average Translational RMSE drift=0.00551, Average Rotational RMSE drift=0.00982, path_lenght=tensor(29.4528, device='cuda:0')]


Модель сохранена


Эпоха валидационная 2/10: 100%|██████████| 910/910 [04:19<00:00,  3.50it/s, loss=2.78e-5, Average Translational RMSE drift=0.00199, Average Rotational RMSE drift=0.00262, path_lenght=tensor(29.4528, device='cuda:0')]


Модель сохранена


Эпоха валидационная 3/10: 100%|██████████| 910/910 [04:22<00:00,  3.46it/s, loss=2.33e-5, Average Translational RMSE drift=0.0016, Average Rotational RMSE drift=0.00233, path_lenght=tensor(29.4528, device='cuda:0')]


...

Модель сохранена


Эпоха валидационная 9/10: 100%|██████████| 910/910 [04:19<00:00,  3.51it/s, loss=0, Average Translational RMSE drift=0, Average Rotational RMSE drift=0, path_lenght=tensor(29.4528, device='cuda:0')]


Модель сохранена


Эпоха валидационная 10/10: 100%|██████████| 910/910 [04:16<00:00,  3.55it/s, loss=0, Average Translational RMSE drift=0, Average Rotational RMSE drift=0, path_lenght=tensor(29.4528, device='cuda:0')]


Модель сохранена
